In [25]:
import sys
import os
import json
import lzma
import numpy as np
import networkx as nx
from tqdm import tqdm
from typing import Hashable, TypeAlias
import gc
from gensim.models import Word2Vec
import umap
import joblib  # for saving the model
import warnings
import hdbscan

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset





In [26]:
CLASSES_PATH = os.path.dirname(os.path.abspath('D:/Code/Classes'))
if not (CLASSES_PATH in sys.path):
    sys.path.append(CLASSES_PATH)
from Classes.Files_Handler_Class import Files_Handler
from Classes.Bcolors_Class import Bcolors as bcolors
from Classes.Random_Walk import Random_Walk
from Classes.Generate_Embedings import Generate_Embedings
from Classes.Load_Graph import Load_Graph
from Classes.Get_Past_Results_Class import Get_Past_Results
from Classes.Embedings_Data_Files_Class import Embedings_Data_Files

In [27]:
load_graph_obj = Load_Graph()
files_handler_obj = Files_Handler()
random_walk_obj = Random_Walk()
embeddings_obj = Generate_Embedings()
# get_past_results_obj = Get_Past_Results()
embedings_data_obj = Embedings_Data_Files()

In [28]:
Node: TypeAlias = Hashable  # Alias for readability

In [29]:
root_path = files_handler_obj.select_dir()
if root_path is None or root_path == '':
    sys.exit(1)
root_path += "/"
print(f"{bcolors.cyan_fg}{bcolors.underline}Root path: {root_path}{bcolors.ENDC}")
networks_extensions = [".edges", ".edgelist", ".mtx", ".gml", ".txt"]
networks_list = files_handler_obj.get_files_by_extensions(root_path, networks_extensions)
print(f"number of networks found: {len(networks_list)}")

Root path: D:/Datasets/CD/Multiplex/Dr Bouyer/LFR/LFR1000/New folder/
number of networks found: 2


In [30]:
task_type = "IM"
embedding_type = 'Vector'
embedding_method = "Frequent Nodes"
embedding_attribute = 'Degree'
compression_method = 'lzma'
umap_n_components = 128
if embedding_attribute == 'Label' and embedding_method == 'Scale':
    sys.exit(0)

walk_length = 128
dimensions = walk_length
walk_depth = 3
if embedding_type == 'Vector':
    num_walks = 1
else:
    num_walks = 64

In [31]:
def load_network_walks(file_info: str, embedding_method: str, embedding_type:str,
                       walk_length: int, walk_depth: int, num_walks: int,
                       labels_counter:int):
    
    result_file_name = file_info['path'] + file_info['name'] + '/'
    result_file_name += f"{file_info['name']} nodes {embedding_method} random walk {embedding_type}"
    result_file_name += f" walk_length=512 walk_depth={walk_depth} num_walks={num_walks}.xz"
    network_walks = None
    if os.path.exists(result_file_name):
        with lzma.open(result_file_name, "rt", encoding="utf-8") as f:
            network_walks = json.load(f)
    
    if network_walks is None:
        print(f"{bcolors.red_fg}{bcolors.bold}Network walks file not found.{bcolors.end_color}")
        print(f"{bcolors.underline}{bcolors.cyan_fg}{result_file_name}{bcolors.end_color}")
        sys.exit()
    network_new_labels = {}
    for i, node in enumerate(network_walks.keys()):
        network_new_labels[node] = str(i + labels_counter)
    new_labels_counter = labels_counter + len(network_walks)

    walks = []
    for k, v in network_walks.items():
        temp_walk = []
        for j in range(walk_length):
            temp_walk.append(network_new_labels[v[j]])
        walks.append(temp_walk)

    return walks, network_new_labels, new_labels_counter

def load_word2vec_model(path='walk2vec.model'):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Model not found at {path}")
    model = Word2Vec.load(path)
    print(f"📥 Model loaded from {path}")
    return model

def train_word2vec(all_walks, embedding_size=32, save_path='Word2vec.model', tabs=''):
    model = Word2Vec(
        sentences=all_walks,
        vector_size=embedding_size,  # size of output embeddings
        window=5,
        min_count=0,
        sg=1,             # skip-gram
        workers=4,
        epochs=5
    )
    model.save(save_path)
    print(f"{tabs}{bcolors.green_fg}Word2vec model trained and saved.{bcolors.ENDC}")
    return model

def walk_to_vector(walk, model):
    vectors = []
    for token in walk:
        if token in model.wv:
            vectors.append(model.wv[token])
        else:
            # Optional: handle unknown tokens
            vectors.append(np.zeros(model.vector_size))
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.concatenate(vectors, axis=0)

def train_UMAP(X, n_components=128, random_state=42, save_path='umap_model.pkl', tabs='\t'):
    reducer = umap.UMAP(n_components=n_components, random_state=random_state)
    umap_model = reducer.fit_transform(X)
    joblib.dump(reducer, save_path)
    print(f"{tabs}{bcolors.green_fg}UMAP model trained and saved.{bcolors.ENDC}")
    return reducer

def make_json_serializable(obj):
    """
    Recursively convert any object (NumPy, PyTorch, etc.) 
    to a JSON-serializable Python type.
    """
    if isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}

    elif isinstance(obj, list):
        return [make_json_serializable(v) for v in obj]

    elif isinstance(obj, tuple):
        return tuple(make_json_serializable(v) for v in obj)

    elif isinstance(obj, np.ndarray):
        return obj.tolist()

    elif isinstance(obj, (np.integer,)):
        return int(obj)

    elif isinstance(obj, (np.floating,)):
        return float(obj)

    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().tolist()

    elif isinstance(obj, (np.bool_, bool)):
        return bool(obj)

    elif obj is None:
        return None

    else:
        # fallback for unsupported objects
        try:
            json.dumps(obj)  # check if serializable
            return obj
        except TypeError:
            return str(obj)



In [32]:
all_walks = []
networks_labels = {}
labels_counter = 0
network_counter = 1
for network_file in networks_list:
    network_infos = files_handler_obj.get_file_info(network_file)
    print(f"{network_counter}- {network_infos['name']}")
    # print(f"\t{bcolors.yellow_fg}Loading graph...{bcolors.end_color}")
    # graph = load_graph_obj.load_monoplex_graph(network_file, network_infos)
    print(f"\t{bcolors.yellow_fg}Loading network randome walks...{bcolors.end_color}")
    network_walks, network_labels, labels_counter = load_network_walks(network_infos, embedding_method, embedding_type,
                                       walk_length, walk_depth, num_walks,
                                       labels_counter)
    print(f"\t{bcolors.green_fg}Loading randome walks done.{bcolors.end_color}")
    all_walks.extend(network_walks)
    networks_labels[network_infos['name']] = network_labels
        
    network_counter += 1
    gc.collect()


1- mu_1_all
	Loading network randome walks...
Network walks file not found.
D:/Datasets/CD/Multiplex/Dr Bouyer/LFR/LFR1000/New folder/a_mu_1/mu_1_all/mu_1_all nodes Frequent Nodes random walk Vector walk_length=512 walk_depth=3 num_walks=1.xz


SystemExit: 

In [ ]:
models_path = files_handler_obj.make_dir(root_path, "Walk2vec Models")
print(f"Models save path: {bcolors.underline}{bcolors.cyan_fg}{models_path}{bcolors.end_color}")


Models save path: D:/Datasets/IM/Monoplex/Public/FACEBOOK NETWORKS/Walk2vec Models/


In [ ]:
# 1. Train Word2vec model
if os.path.exists(models_path + 'walk2vec.model'):
    print(f"{bcolors.yellow_fg}Load existing Word2vec model...{bcolors.end_color}")
    word2vec_model = Word2Vec.load(models_path + 'walk2vec.model')
    print(f"{bcolors.green_fg}Word2vec Model loading done.{bcolors.end_color}")
else:
    print(f"{bcolors.yellow_fg}Train word2vec model...{bcolors.end_color}")
    word2vec_model = train_word2vec(all_walks, save_path= models_path + 'walk2vec.model')



Load existing Word2vec model...
Word2vec Model loading done.


In [ ]:
# Generate Word2vec Embeddings
word2vec_walks = []
for walk in all_walks:
    word2vec_walks.append(walk_to_vector(walk, word2vec_model))
word2vec_walks = np.array(word2vec_walks)
word2vec_walks.shape

(450891, 4096)

In [ ]:
del all_walks
gc.collect()

22

In [ ]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, input_dim=4096, latent_dim=128):
        super(Conv1DAutoencoder, self).__init__()

        # ---------- Encoder ----------
        # Input [B, 1, 4096]
        # Output [B, 256, 128]
        self.encoder = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=16, kernel_size=5, stride=2, padding=2),  # [B, 16, 2048]
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.Conv1d(16, 32, kernel_size=5, stride=2, padding=2),  # [B, 32, 1024]
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=5, stride=2, padding=2),  # [B, 64, 512]
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2), # [B, 128, 256]
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 256, kernel_size=5, stride=2, padding=2),# [B, 256, 128]
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )

        # ---------- FC Encoder ----------
        # Input [B, 32768]
        # Output [B, 128]
        self.fc_enc = nn.Linear(256 * (input_dim // (2**5)), latent_dim)  # 4096 → 128

        # ---------- FC Decoder ----------
        # Input [B, 128]
        # Output [B, 32768]
        self.fc_dec = nn.Linear(latent_dim, 256 * (input_dim // (2**5)))

        # ---------- Decoder ----------
        # Input [B, 256, 128]
        # Output [B, 1, 4096]
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(in_channels=256, out_channels=128, kernel_size=5, stride=2, padding=2, output_padding=1),# [B, 256, 128]
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.ConvTranspose1d(128, 64, kernel_size=5, stride=2, padding=2, output_padding=1),# [B, 128, 64]
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.ConvTranspose1d(64, 32, kernel_size=5, stride=2, padding=2, output_padding=1),# [B, 64, 32]
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.ConvTranspose1d(32, 16, kernel_size=5, stride=2, padding=2, output_padding=1),# [B, 32, 16]
            nn.BatchNorm1d(16),
            nn.ReLU(),

            nn.ConvTranspose1d(16, 1, kernel_size=5, stride=2, padding=2, output_padding=1),# [B, 16, 1]
            nn.Sigmoid()  # normalize to [0,1]
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # [B, 1, 4096]
        z = self.encoder(x) # [B, 256, 128]
        z_flat = z.view(z.size(0), -1) # [B, 32768]
        latent = self.fc_enc(z_flat) # [B, 128]
        z_dec = self.fc_dec(latent) # [B, 32768]
        z_dec = z_dec.view(x.size(0), 256, -1) # [B, 256, 128]
        out = self.decoder(z_dec) # [B, 1, 4096]
        return out.squeeze(1), latent
    


In [ ]:
class Conv1DAutoencoder(nn.Module):
    def __init__(self, input_dim=4096, latent_dim=128):
        super(Conv1DAutoencoder, self).__init__()

        # ---------- Encoder ----------
        # Input [B, 1, 4096]
        # Output [B, 256, 128]
        self.encoder_conv = nn.Sequential(
            nn.Conv1d(
                in_channels=1,
                out_channels=256,
                kernel_size=32,
                stride=32,
                padding=0  # no padding
            ),
            nn.BatchNorm1d(256),
            nn.ReLU()
        )

        # ---------- FC Encoder ----------
        self.fc_enc = nn.Linear(256 * (input_dim // 32), latent_dim)  # 256*128 → 128 latent

        # ---------- FC Decoder ----------
        self.fc_dec = nn.Linear(latent_dim, 256 * (input_dim // 32))  # 128 → 256*128

        # ---------- Decoder ----------
        # Input [B, 256, 128]
        # Output [B, 1, 4096]
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose1d(
                in_channels=256,
                out_channels=1,
                kernel_size=32,
                stride=32,
                padding=0
            ),
            nn.Sigmoid()  # to normalize output [0,1]
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # [B, 1, 4096]

        # Encode
        z = self.encoder_conv(x)              # [B, 256, 128]
        z_flat = z.view(z.size(0), -1)        # [B, 256*128]
        latent = self.fc_enc(z_flat)         # [B, latent_dim]

        # Decode
        z_dec = self.fc_dec(latent)          # [B, 256*128]
        z_dec = z_dec.view(x.size(0), 256, -1)  # [B, 256, 128]
        out = self.decoder_conv(z_dec)       # [B, 1, 4096]

        return out.squeeze(1), latent



In [ ]:
def train_convae(model, dataloader, loss_fn, optimizer, save_path,
                 epochs=50, lr=1e-3, device='cuda'):

    model.to(device)
    best_loss = np.inf

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        progress_bar = tqdm(dataloader, desc=f"\tEpoch {epoch+1}", leave=True)
        for batch in progress_bar:
            x = batch[0].to(device)

            optimizer.zero_grad()
            reconstructed, _ = model(x)
            loss = loss_fn(reconstructed, x)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        avg_loss = running_loss / len(dataloader)
        if avg_loss <= best_loss:
            best_loss = avg_loss
            torch.save(model.state_dict(), save_path+"ConvAE best loss.pt")
            print(f"\t{bcolors.green_fg}Avg Loss: {avg_loss:.6f}{bcolors.ENDC}")
        else:
            print(f"\t{bcolors.red_fg}Avg Loss: {avg_loss:.6f}{bcolors.ENDC}")

    torch.save(model.state_dict(), save_path+"ConvAE.pt")

def load_convae(model, load_path='convae.pth', device='cpu'):
    if os.path.exists(load_path):
        model.load_state_dict(torch.load(load_path, map_location=device))
        model.to(device)
        model.eval()
        print(f"{bcolors.green_fg}Model file found at:{bcolors.ENDC}" +
               f"{bcolors.underline}{bcolors.cyan_fg}{load_path}{bcolors.ENDC}")
    else:
        model = None
        print(f"{bcolors.red_fg}Model file not found at:{bcolors.ENDC}" +
               f"{bcolors.underline}{bcolors.cyan_fg}{load_path}{bcolors.ENDC}")
    # print(f"✅ ConvAE model loaded successfully from: {load_path}")
    return model

In [ ]:
# Train ConvAE Model

input_dim = len(word2vec_walks[0])
latent_dim = walk_length
batch_size = 512
epochs = 100
lr=1e-3
device ='cuda' if torch.cuda.is_available() else 'cpu'

# Initialize model
model = Conv1DAutoencoder(input_dim=input_dim, latent_dim=latent_dim)

# Load trained model
model = load_convae(model, models_path+'ConvAE.pt', device)
if model == None:
    word2vec_walks = torch.from_numpy(word2vec_walks).to(torch.float32)
    dataset = TensorDataset(word2vec_walks)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Initialize model
    model = Conv1DAutoencoder(input_dim=input_dim, latent_dim=latent_dim)

    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Train ConvAE
    train_convae(model, dataloader, loss_fn, optimizer, save_path=models_path, epochs=epochs, lr=lr, device=device, )



Model file found at:D:/Datasets/IM/Monoplex/Public/FACEBOOK NETWORKS/Walk2vec Models/ConvAE.pt


In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

labels_counter = 0
network_counter = 1
for network_file in networks_list:
    network_infos = files_handler_obj.get_file_info(network_file)
    save_path = network_infos['path'] + network_infos['name'] + '/'
    print(f"{network_counter}- {network_infos['name']}")

    next_labels_counter = labels_counter + len(networks_labels[network_infos['name']])

    print(f"\t{bcolors.yellow_fg}Encode to 128-d embeddings by Conv1DAutoencoder...{bcolors.end_color}")
    print(f"\tStart index: {bcolors.yellow_fg}{labels_counter}{bcolors.ENDC}, " +
          f"End index: {bcolors.yellow_fg}{next_labels_counter}{bcolors.ENDC}")
    # Encode to 128-d embeddings
    with torch.no_grad():
        temp_word2vec_walks = torch.from_numpy(word2vec_walks[labels_counter:next_labels_counter]).to(torch.float32)
        _, final_embeddings = model(temp_word2vec_walks.to(device))
    print("\tCompressed embeddings shape:", final_embeddings.shape)
    del temp_word2vec_walks
    gc.collect()

    print(f"\t{bcolors.yellow_fg}Clustering by HDBSCAN...{bcolors.end_color}")
    # 4. Clustering with HDBSCAN
    clusterer = hdbscan.HDBSCAN(min_cluster_size=5, metric='euclidean')
    labels = clusterer.fit_predict(final_embeddings)

    print(f"\tPrepare final data...{bcolors.end_color}")
    network_final_embeddings ={}
    for c, node in enumerate(networks_labels[network_infos['name']].keys()):
        network_final_embeddings[node] = {}
        # network_final_embeddings[node]['new_lbl'] = networks_labels[network_infos['name']][node]
        network_final_embeddings[node]['emb'] = final_embeddings[c].tolist()
        network_final_embeddings[node]['class'] = labels[c]
    
    print(f"\t{bcolors.yellow_fg}Write data in:{bcolors.ENDC}{bcolors.underline}{bcolors.cyan_fg}{save_path}{bcolors.ENDC}")
    with lzma.open(save_path + network_infos['name'] + " Word2vec ConvAE HDBSCAN Nodes Embedding and Classes.xz", "wt", encoding="utf-8") as f:
        json.dump(make_json_serializable(network_final_embeddings), f)

    with lzma.open(save_path + network_infos['name'] + " Nodes label mapper.xz", "wt", encoding="utf-8") as f:
        json.dump(make_json_serializable(networks_labels[network_infos['name']]), f)
    
    labels_counter = next_labels_counter
    network_counter += 1
    gc.collect()

1- socfb-wosn-friends(63731 N and 817090 E)
	Encode to 128-d embeddings by Conv1DAutoencoder...
	Start index: 0, End index: 63731
	Compressed embeddings shape: torch.Size([63731, 128])
	Clustering by HDBSCAN...
	Prepare final data...
	Write data in:D:/Datasets/IM/Monoplex/Public/FACEBOOK NETWORKS/10K - 100K/socfb-wosn-friends(63731 N and 817090 E)/
2- socfb-Harvard1(15126 N and 824617 E)
	Encode to 128-d embeddings by Conv1DAutoencoder...
	Start index: 63731, End index: 78857
	Compressed embeddings shape: torch.Size([15126, 128])
	Clustering by HDBSCAN...
	Prepare final data...
	Write data in:D:/Datasets/IM/Monoplex/Public/FACEBOOK NETWORKS/10K - 100K/socfb-Harvard1(15126 N and 824617 E)/
3- socfb-Michigan23(30147 N and 1176516 E)
	Encode to 128-d embeddings by Conv1DAutoencoder...
	Start index: 78857, End index: 109004
	Compressed embeddings shape: torch.Size([30147, 128])
	Clustering by HDBSCAN...
	Prepare final data...
	Write data in:D:/Datasets/IM/Monoplex/Public/FACEBOOK NETWORKS/